In [ ]:
# Code adapted from a CS4051 tutorial

import torch

import pickle

from torch.nn.functional import log_softmax

from transformers import AutoTokenizer, AutoModelForCausalLM



def load_model(hf_id: str, device: "torch.device", hf_token: str) -> tuple:

    tokenizer = AutoTokenizer.from_pretrained(hf_id, token=hf_token)

    model = AutoModelForCausalLM.from_pretrained(hf_id, token=hf_token).to(device)

    return tokenizer, model



def get_log_p_tws(

    context,
    target,

    tokenizer,

    model,

    device: "torch.device" = torch.device("cpu"),

    verbose=False,
):
    """

    Compute sentence-level, word-level, and token-level log probability

    of a target sentence given a prime sentence.


    Args:
    =====

    * `context` (str): The context sentence text.

    * `target` (str): The target sentence text.

    * `tokenizer` (AutoTokenizer): The (HuggingFace) tokenizer used to encode the input.

    * `model` (AutoModelForCausalLM): The (HuggingFace) model used to generate the logits.

    * `device` (torch.device, optional): The device to run the model on. Defaults to torch.device('cpu').


    Returns:
    ========

    Dict[str, Union[float, List[float]]]:

        A dictionary containing the sentence-level, word-level,

        and token-level log probabilities of the target sentence.

        E.g. {'s': ..., 'w': [..., ...], 't': [..., ...]}
    """

    if verbose:

        print("context:", context)
        print("target:", target)

    input_str = f"{tokenizer.eos_token} {context} {target}".replace("  ", " ")

    if verbose:
        print("input str:", input_str)


    input_ids = tokenizer.encode(input_str, return_tensors="pt").to(device)

    if verbose:
        print("input ids:", input_ids)


    # useful if we want to do conditional generation: get target start index

    target_start_idx = len(

        tokenizer.encode(tokenizer.eos_token + context, return_tensors="pt")[0]
    )


    if target_start_idx >= len(input_ids[0]):

        target_start_idx = len(input_ids[0]) - 1


    if verbose:

        print("target start idx:", target_start_idx)

    if verbose:

        print("target start token:", tokenizer.decode(input_ids[0][target_start_idx]))


    # get logits and probs of target tokens

    with torch.no_grad():

        generated = model(input_ids)

    logits = generated.logits[0][target_start_idx - 1 : -1]

    probs = log_softmax(logits, dim=-1)

    if verbose:
        print("len logits:", len(logits))

        print("len probs:", len(probs))


    target_token_ids = input_ids[0][target_start_idx:].tolist()

    target_probs = probs[range(probs.size(0)), target_token_ids]

    if verbose:
        print("target token ids:", target_token_ids)

        print("target probs:", target_probs)


    target_word_probs = [target_probs[0].item()]

    token_strs = tokenizer.convert_ids_to_tokens(target_token_ids)

    if verbose:
        print("token strs:", token_strs)

    # We start from 1 because the first token has already been added above

    for i, token_str in enumerate(token_strs[1:], start=1):

        if not (token_str.startswith("Ġ") or token_str == "."):

            target_word_probs[-1] += target_probs[i].item()
        else:

            target_word_probs.append(target_probs[i].item())


    if verbose:

        print(target_word_probs)


    target_sent_logp = torch.mean(target_probs).item()

    target_tkns_logp = target_probs.tolist()


    if verbose:
        print(target_tkns_logp, target_sent_logp)

    return {

        "s": target_sent_logp,  # sentence

        "w": target_word_probs,  # words

        "t": target_tkns_logp,  # tokens

    }



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "meta-llama/Llama-3.2-1B"

hf_token = ""  # Add your Hugging Face token here

tokenizer, model = load_model(MODEL_NAME, device, hf_token)

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [ ]:
topics_per_level = pickle.load(open("pickle/topics_per_level.pkl", "rb"))
topics_per_level

FileNotFoundError: [Errno 2] No such file or directory: 'topics_per_level.pkl'

In [ ]:
topics_per_level = pickle.load(open("pickle/topics_per_level.pkl", "rb"))

topic_surprisals_per_level = {1: {}, 2: {}, 3: {}, 4: {}}
for level, samples in topics_per_level.items():
    for topic, samples_list in samples.items():
        topic_surprisals_per_level[level][topic] = []
        for sample in samples_list:
            context = sample["Context"]
            target = sample["Tutor"]
            if context:
                log_p = get_log_p_tws(
                    context, target, tokenizer, model, device, verbose=False
                )["s"]
                topic_surprisals_per_level[level][topic].append(log_p)

with open("pickle/topic_surprisals_per_level.pkl", "wb") as f:
    pickle.dump(topic_surprisals_per_level, f)

In [ ]:
all_grouped_by_level = pickle.load(open("pickle/all_grouped_by_level.pkl", "rb"))

In [4]:
all_grouped_by_level

In [ ]:
all_tutor_surprisals_per_level = {1: [], 2: [], 3: [], 4: []}
for level, group in all_grouped_by_level:
    for i, utt in group.iterrows():
        if utt["PrevTurn"] is not None and utt["Participant"] == "INV":
            context = utt["PrevTurn"]
            target = utt["Turn"]
            log_p = get_log_p_tws(
                context, target, tokenizer, model, device, verbose=False
            )["s"]
            all_tutor_surprisals_per_level[level].append(log_p)

In [ ]:
with open("pickle/all_tutor_surprisals_per_level.pkl", "wb") as file:
    pickle.dump(all_tutor_surprisals_per_level, file)

In [ ]:
example_dialogue = pickle.load(open("pickle/example_dialogue.pkl", "rb"))

example_dialogue["Surprisal"] = example_dialogue.apply(
    lambda row: (
        get_log_p_tws(
            row["PrevTurn"], row["Turn"], tokenizer, model, device, verbose=False
        )["s"]
        if row["PrevTurn"] is not None
        else None
    ),
    axis=1,
)

In [ ]:
with open("pickle/example_dialogue_with_surprisal_new.pkl", "wb") as file:
    pickle.dump(example_dialogue, file)

In [ ]:
miami_all_utts = pickle.load(open("pickle/miami_all_utts.pkl", "rb"))
miami_all_surprisals = []

for i, utt in miami_all_utts.iterrows():
    if utt["PrevTurn"] is not None:
        context = utt["PrevTurn"]
        target = utt["Turn"]
        log_p = get_log_p_tws(context, target, tokenizer, model, device, verbose=False)[
            "s"
        ]
        miami_all_surprisals.append(log_p)

In [ ]:
with open("pickle/miami_all_surprisals.pkl", "wb") as file:
    pickle.dump(miami_all_surprisals, file)

In [ ]:
miami_all_switched_utts = pickle.load(open("pickle/miami_all_switched_utts.pkl", "rb"))
miami_all_switched_surprisals = []

for i, utt in miami_all_switched_utts.iterrows():
    if utt["PrevTurn"] is not None:
        context = utt["PrevTurn"]
        target = utt["Turn"]
        log_p = get_log_p_tws(context, target, tokenizer, model, device, verbose=False)[
            "s"
        ]
        miami_all_switched_surprisals.append(log_p)

In [ ]:
with open("pickle/miami_switched_surprisals.pkl", "wb") as file:
    pickle.dump(miami_all_switched_surprisals, file)

In [ ]:
survey_samples_formatted = pickle.load(
    open("pickle/survey_samples_formatted.pkl", "rb")
)

survey_samples_formatted

{'backchannel': [{'Tutor': 'because hm classes', 'Context': 'sí'},
  {'Tutor': 'okay okay', 'Context': 'es a ver I play in juveniles'},
  {'Tutor': "it's difficult or it's easy", 'Context': 'pues'},
  {'Tutor': 'yeah okay',
   'Context': 'hm and bueno the sundays I make homework'},
  {'Tutor': 'homework okay do you watch tv', 'Context': 'sí'},
  {'Tutor': 'you understand did you understand what I said no',
   'Context': 'hm pues no'},
  {'Tutor': 'yes', 'Context': 'si tinc germans'},
  {'Tutor': 'films', 'Context': 'hm no a ver espera hm pelí bueno películas'},
  {'Tutor': "don't worry little by little",
   'Context': 'well I bueno es que estoy saturado'},
  {'Tutor': "what don't you like", 'Context': 'no sé qué me gusta'},
  {'Tutor': 'armari', 'Context': 'right hm armari'},
  {'Tutor': 'no ho entès', 'Context': 'ah hm old'},
  {'Tutor': 'alguna otra', 'Context': 'fine'},
  {'Tutor': 'quines coses té', 'Context': 'and'},
  {'Tutor': 'sí sí dime dime', 'Context': "I'd like"},
  {'Tutor

In [ ]:
survey_samples_surprisals = {}
for intent, samples in survey_samples_formatted.items():
    survey_samples_surprisals[intent] = []
    for sample in samples:
        context = sample["Context"]
        target = sample["Tutor"]
        if context:
            log_p = get_log_p_tws(
                context, target, tokenizer, model, device, verbose=False
            )["s"]
            survey_samples_surprisals[intent].append(
                {"Context": context, "Tutor": target, "Surprisal": log_p}
            )

with open("pickle/survey_samples_surprisals.pkl", "wb") as f:
    pickle.dump(survey_samples_surprisals, f)